# Camada Bronze: Ingestão de Dados Poloniex OHLCV

## Objetivo

Ingerir candles horários de OHLCV da API pública da Poloniex para BTC, ETH, SOL, ADA e LINK.

## Estratégia de Ingestão

- **D-1 Incremental:** Coleta apenas o dia anterior (D-1) a cada execução
- **Cadência:** Agendado para rodar uma vez por dia
- **Idempotência:** Pode rodar múltiplas vezes no mesmo dia sem duplicar dados
- **Backfill:** Suporta reprocessamento de dias específicos alterando TARGET_DATE
- **Imutabilidade:** Candles OHLCV são imutáveis após fechamento do período

**Destino:** `uc_sa_br_dev.bronze_poloniex_ohlcv.hourly`

## Princípios da Camada Bronze

- Selecionar campos relevantes (descartar campos desnecessários da API)
- Manter tipos de dados originais (sem casting, sem conversões)
- Adicionar metadata para auditoria: `ingested_at`
- Adicionar `rate_date` para particionamento físico (otimização de storage)
- Adiar transformações de negócio para camada Silver
- Identificação da fonte implícita no nome do schema

## Campos Armazenados

- **Campos da API:** symbol, low, high, open, close, volume, start_time_ms, close_time_ms, interval, trade_count (tipos originais)
- **Metadata de Auditoria:** ingested_at (timestamp do processo)
- **Partição Física:** rate_date (derivado de start_time_ms)
- **Fonte:** Implícita no nome do schema (bronze_poloniex_ohlcv)

## Benefícios da Estratégia D-1 Incremental

- **Performance:** ~95% menos dados escritos por execução (120 vs 2500 registros)
- **Custo:** API calls menores, menos compute time, redução de transferência de dados
- **Idempotência:** Múltiplas execuções no mesmo dia não duplicam dados
- **Backfill Preciso:** Reprocessar dias específicos sem afetar histórico
- **Resiliência:** Proteção contra falhas com retry seguro

## Adição de Transformações

Transformações de negócio (type casting, conversão de timestamp, validações de qualidade) são adiadas para camada Silver. O campo `exchange` será adicionado na Silver para analytics multi-exchange.

In [0]:
import json
import logging
import sys
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone, date

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

ZORDON_SRC_PATH = "/Workspace/Repos/CryptoLake/zordon-data-utils/src"
if ZORDON_SRC_PATH not in sys.path:
    sys.path.append(ZORDON_SRC_PATH)

import zordon
from pyspark.sql import functions as F

In [0]:
SYMBOLS = ["BTC_USDT", "ETH_USDT", "SOL_USDT", "ADA_USDT", "LINK_USDT"]
INTERVAL = "HOUR_1"
TABLE_NAME = "hourly"
MAX_LIMIT = 500

TARGET_DATE = date.today() - timedelta(days=1)

logging.info(f"Configuration loaded - Target date: {TARGET_DATE}")

In [0]:
def utc_now_ms():
    """Timestamp atual em milissegundos UTC."""
    return int(datetime.now(tz=timezone.utc).timestamp() * 1000)


def utc_days_ago_ms(days):
    """Timestamp de N dias atras em milissegundos UTC."""
    dt = datetime.now(tz=timezone.utc) - timedelta(days=days)
    return int(dt.timestamp() * 1000)

In [0]:
def fetch_poloniex_candles(symbol, start_time_ms, end_time_ms, limit=MAX_LIMIT):
    """Fetch hourly candles from Poloniex public API."""
    base_url = f"https://api.poloniex.com/markets/{symbol}/candles"
    params = urllib.parse.urlencode({
        "interval": INTERVAL,
        "startTime": start_time_ms,
        "endTime": end_time_ms,
        "limit": limit,
    })
    url = f"{base_url}?{params}"

    with urllib.request.urlopen(url, timeout=30) as response:
        if response.status != 200:
            raise RuntimeError(f"Poloniex API error: HTTP {response.status}")
        payload = response.read().decode("utf-8")

    return json.loads(payload)


def parse_candle(symbol, row):
    """Extract relevant fields for CryptoLake project.

    Pragmatic Bronze approach:
    - Select only fields needed for OHLCV analysis
    - Keep original data types (no casting, no conversions)
    - Discard fields not used in the project (buyTakerAmount, weightedAverage, etc.)

    Poloniex API response (14 fields):
    [0] low, [1] high, [2] open, [3] close, [4] amount, [5] quantity,
    [6] buyTakerAmount, [7] buyTakerQuantity, [8] tradeCount, [9] ts,
    [10] weightedAverage, [11] interval, [12] startTime, [13] closeTime.
    """
    return {
        "symbol": symbol,
        "low": row[0],
        "high": row[1],
        "open": row[2],
        "close": row[3],
        "volume": row[5],
        "start_time_ms": row[12],
        "close_time_ms": row[13],
        "interval": row[11],
        "trade_count": row[8],
    }

In [0]:
def collect_hourly_candles(symbols, target_date):
    """Collect hourly candles for multiple symbols for a specific date.

    Args:
        symbols: List of symbol pairs (e.g., ["BTC_USDT", "ETH_USDT"])
        target_date: Date to collect data for (date object)

    Returns:
        List of raw candle records with metadata.

    Note:
        D-1 Incremental Strategy: Collects only 24 hourly candles per symbol.
        For backfill, pass a specific date. For normal operation, use D-1.
        Current limitation: MAX_LIMIT (500) is sufficient for 1-day collection.
    """
    # Calculate start and end timestamps for the target date (00:00 to 23:59:59 UTC)
    start_time_ms = int(
        datetime.combine(target_date, datetime.min.time(), timezone.utc).timestamp() * 1000
    )
    end_time_ms = int(
        datetime.combine(target_date, datetime.max.time(), timezone.utc).timestamp() * 1000
    )

    all_records = []

    for symbol in symbols:
        try:
            logging.info(f"Fetching {symbol} candles...")
            candles = fetch_poloniex_candles(
                symbol=symbol,
                start_time_ms=start_time_ms,
                end_time_ms=end_time_ms,
                limit=MAX_LIMIT,
            )

            for candle_row in candles:
                record = parse_candle(symbol, candle_row)
                all_records.append(record)

            logging.info(f"Collected {len(candles)} candles for {symbol}")

        except Exception as e:
            logging.error(f"Failed to fetch {symbol}: {e}")
            raise

    return all_records

In [0]:
proj = zordon.Project(
    spark=spark,
    country="br",
    region="sa",
    environment="dev",
)

bronze_poloniex = proj.client(
    layer="bronze",
    domain="poloniex",
    subdomain="ohlcv",
)

target_fqn = bronze_poloniex.governance.fqn(TABLE_NAME)
valid = zordon.is_valid_name(TABLE_NAME)

logging.info(f"Target table: {target_fqn}")
logging.info(f"Table name valid: {valid}")

In [0]:
logging.info(f"Starting D-1 incremental collection for date: {TARGET_DATE}")
logging.info(f"Symbols: {SYMBOLS}")

records = collect_hourly_candles(SYMBOLS, TARGET_DATE)

logging.info(f"Total records collected: {len(records)} ({len(records) // len(SYMBOLS)} per symbol)")

if not records:
    raise RuntimeError("No records returned from Poloniex API")

In [0]:
df = (
    spark.createDataFrame(records)
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn(
        "rate_date",
        F.to_date(F.from_unixtime(F.col("start_time_ms") / 1000))
    )
)

total_rows = df.count()
logging.info(f"DataFrame created with {total_rows} records")

In [0]:
partitions_in_df = df.select("rate_date").distinct().collect()
partition_dates = sorted([row.rate_date for row in partitions_in_df])
logging.info(f"Writing {len(partition_dates)} partition(s): {partition_dates}")

written_fqn = bronze_poloniex.write_table(
    df=df,
    table_name=TABLE_NAME,
    mode="overwrite",
    partition_cols=["rate_date"],
    dynamic_partition_overwrite=True,
)

logging.info(f"Bronze table written: {written_fqn}")

In [0]:
df_read = bronze_poloniex.read_table(TABLE_NAME)

validation = (
    df_read
    .groupBy("symbol")
    .agg(
        F.count("*").alias("rows"),
        F.min("rate_date").alias("min_date"),
        F.max("rate_date").alias("max_date"),
    )
    .orderBy("symbol")
    .collect()
)

for row in validation:
    logging.info(f"Symbol: {row.symbol} | Rows: {row.rows} | Date range: {row.min_date} to {row.max_date}")

logging.info("Bronze ingestion completed successfully")

## Backfill: Reprocessamento de Dias Específicos

A estratégia D-1 incremental permite reprocessar dias específicos sem perder histórico.

### Caso 1: Reprocessar Um Dia Específico
```python
# Na célula Configuration, alterar TARGET_DATE:
TARGET_DATE = date(2026, 6, 20)  # Reprocessar 2026-06-20

# Executar células 9-11
# Resultado: apenas a partição 2026-06-20 será reescrita
```

### Caso 2: Backfill de Múltiplos Dias
```python
start_date = date(2026, 6, 15)
end_date = date(2026, 6, 20)

current_date = start_date
while current_date <= end_date:
    logging.info(f"Processing {current_date}")
    records = collect_hourly_candles(SYMBOLS, current_date)
    df = (
        spark.createDataFrame(records)
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("rate_date", F.to_date(F.from_unixtime(F.col("start_time_ms") / 1000)))
    )
    bronze_poloniex.write_table(
        df, TABLE_NAME, 
        mode="overwrite", 
        partition_cols=["rate_date"], 
        dynamic_partition_overwrite=True
    )
    current_date += timedelta(days=1)
```

### Caso 3: Detecção de Lacunas
```python
# Verificar quais datas estão na tabela
df_dates = (
    bronze_poloniex.read_table(TABLE_NAME)
    .select("rate_date")
    .distinct()
    .orderBy("rate_date")
)
display(df_dates)

# Identificar lacunas e reprocessar dias faltantes
```